# Inference — EQCCT

Runs the baseline and fine-tuned EQCCT models on the configured splits and stores raw records for later evaluation. It does not calculate final metrics.


## 1. Environment

In [1]:
import sys
REPO = r"M:\Github\TFM_SeismicPhasePicker_ColombianAndesRegion"
sys.path.insert(0, REPO + r"\src\training")     # training_utils
sys.path.insert(0, REPO + r"\src\evaluation")   # evaluation_utils

import training_utils as tu
import evaluation_utils as eu

import torch
import numpy as np
import seisbench.models as sbm

tu.banner()

seisbench 0.12.3 | torch 2.5.1+cu121 | device: cuda


## 2. Configuration

In [2]:
ARCH       = "EQCCT"
STATIONS   = None
PRETRAINED = "original"        # SeisBench ships no INSTANCE weights for EQCCT
FT_P       = tu.MODELS_DIR / "eqcctp_ft.pt"
FT_S       = tu.MODELS_DIR / "eqccts_ft.pt"
SIGMA      = 20
IN_SAMPLES = 6000
NORM       = "std"
BATCH      = 64

COLLECT_DEV = True       # also collect dev (needed to choose the threshold)

TAG = tu.make_tag(ARCH, STATIONS)
assert FT_P.exists() and FT_S.exists(), "Missing eqcctp_ft.pt / eqccts_ft.pt (run finetune_eqcct.ipynb first)"
print("TAG:", TAG, "| stations:", "ALL" if not STATIONS else STATIONS)

TAG: eqcct_all | stations: ALL


## 3. Data

In [3]:
data = tu.load_dataset(STATIONS)
train, dev, test = data.train_dev_test()
for name, sub in (("train", train), ("dev", dev), ("test", test)):
    tu.split_summary(sub, name)

No filter (all stations): 106580 traces
stations: ['BRJC', 'CBOC', 'CHI', 'DBB', 'GUY2C', 'JAMC', 'MAL', 'NOR', 'ORTC', 'PAL', 'PIZC', 'PRA', 'PTB', 'PTGC', 'QUBC', 'RUS', 'SPBC', 'TAM', 'URE', 'URMC', 'YOT', 'ZAR']
  train: 74600  {'earthquake': 37300, 'noise': 37300}
  dev  : 16414  {'earthquake': 8207, 'noise': 8207}
  test : 15566  {'earthquake': 7783, 'noise': 7783}


## 4. Models: baseline and fine-tuned

In [4]:
base_p = sbm.EQCCTP.from_pretrained(PRETRAINED).to(tu.DEVICE)
base_s = sbm.EQCCTS.from_pretrained(PRETRAINED).to(tu.DEVICE)
ft_p   = sbm.EQCCTP.from_pretrained(PRETRAINED).to(tu.DEVICE)
ft_s   = sbm.EQCCTS.from_pretrained(PRETRAINED).to(tu.DEVICE)
ft_p.load_state_dict(torch.load(FT_P, map_location=tu.DEVICE))
ft_s.load_state_dict(torch.load(FT_S, map_location=tu.DEVICE))
print("models loaded (two single-phase networks per version)")

MODEL_BASE, MODEL_FT = (base_p, base_s), (ft_p, ft_s)

c:\Users\moise\miniconda3\envs\tesismaster\Lib\site-packages\seisbench\models\base.py:952: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_weights = torch.load(f"{path_p

models loaded (two single-phase networks per version)


## 5. Output decoder

In [5]:
def decode_ps(models, x):
    """EQCCT: two independent single-phase networks, one per phase."""
    mp, ms = models
    P = mp(x)[:, 0].cpu().numpy()   # (B, W)
    S = ms(x)[:, 0].cpu().numpy()
    return P, S

## 6. Resumable inference

Collect baseline and fine-tuned predictions in saved chunks. A parameter manifest prevents reuse under changed settings. Each record stores pseudo-arrival references, predicted peaks, and metadata needed for evaluation.

In [6]:
CHUNK      = 2000       # windows per chunk; lower it for more frequent save points
EXTRA_COLS = ("trace_synthetic", "source_depth_km", "trace_name", "split")

PARTS = tu.RESULTS_DIR / "_parts"

def run(subset, name):
    print(f"\n=== {name}: {len(subset)} windows")
    out = []
    for label, model in (("baseline", MODEL_BASE), ("finetuned", MODEL_FT)):
        print(f"  [{label}]")
        out.append(eu.collect_resumable(
            model, subset, decode_ps,
            part_dir=PARTS / f"{TAG}_{name}" / label,
            chunk=CHUNK, in_samples=IN_SAMPLES, sigma=SIGMA,
            norm_type=NORM, batch_size=BATCH, extra_cols=EXTRA_COLS))
    return out

rec_base, rec_ft = run(test, "test")


=== test: 15566 windows
  [baseline]
  15566 windows in 8 chunks of 2000 | already stored: 0
    chunk 1/8  [0:2000]  0.1 min elapsed, ~1.0 min left
    chunk 2/8  [2000:4000]  0.3 min elapsed, ~0.9 min left
    chunk 3/8  [4000:6000]  0.4 min elapsed, ~0.7 min left
    chunk 4/8  [6000:8000]  0.6 min elapsed, ~0.6 min left
    chunk 5/8  [8000:10000]  0.7 min elapsed, ~0.4 min left
    chunk 6/8  [10000:12000]  0.8 min elapsed, ~0.3 min left
    chunk 7/8  [12000:14000]  1.0 min elapsed, ~0.1 min left
    chunk 8/8  [14000:15566]  1.1 min elapsed, ~0.0 min left
  done: 15566 windows
  [finetuned]
  15566 windows in 8 chunks of 2000 | already stored: 0
    chunk 1/8  [0:2000]  0.1 min elapsed, ~0.8 min left
    chunk 2/8  [2000:4000]  0.2 min elapsed, ~0.7 min left
    chunk 3/8  [4000:6000]  0.3 min elapsed, ~0.6 min left
    chunk 4/8  [6000:8000]  0.5 min elapsed, ~0.5 min left
    chunk 5/8  [8000:10000]  0.6 min elapsed, ~0.3 min left
    chunk 6/8  [10000:12000]  0.7 min elapsed

In [7]:
if COLLECT_DEV:
    dev_base, dev_ft = run(dev, "dev")


=== dev: 16414 windows
  [baseline]
  16414 windows in 9 chunks of 2000 | already stored: 0
    chunk 1/9  [0:2000]  0.1 min elapsed, ~1.1 min left
    chunk 2/9  [2000:4000]  0.3 min elapsed, ~1.0 min left
    chunk 3/9  [4000:6000]  0.4 min elapsed, ~0.9 min left
    chunk 4/9  [6000:8000]  0.6 min elapsed, ~0.7 min left
    chunk 5/9  [8000:10000]  0.7 min elapsed, ~0.6 min left
    chunk 6/9  [10000:12000]  0.8 min elapsed, ~0.4 min left
    chunk 7/9  [12000:14000]  1.0 min elapsed, ~0.3 min left
    chunk 8/9  [14000:16000]  1.1 min elapsed, ~0.1 min left
    chunk 9/9  [16000:16414]  1.1 min elapsed, ~0.0 min left
  done: 16414 windows
  [finetuned]
  16414 windows in 9 chunks of 2000 | already stored: 0
    chunk 1/9  [0:2000]  0.1 min elapsed, ~1.0 min left
    chunk 2/9  [2000:4000]  0.2 min elapsed, ~0.8 min left
    chunk 3/9  [4000:6000]  0.3 min elapsed, ~0.7 min left
    chunk 4/9  [6000:8000]  0.5 min elapsed, ~0.6 min left
    chunk 5/9  [8000:10000]  0.6 min elapsed,

## 7. Save

In [8]:
config = {"arch": ARCH, "stations": STATIONS, "pretrained": PRETRAINED,
          "sigma": SIGMA, "in_samples": IN_SAMPLES, "norm": NORM,
          "peak_floor": 0.05, "peak_dist": 50,
          "subsample_fraction": tu.SUBSAMPLE_FRACTION}

eu.save_records(TAG, ARCH, rec_base, rec_ft, {**config, "split": "test"})

if COLLECT_DEV:
    eu.save_records(TAG + "_dev", ARCH, dev_base, dev_ft, {**config, "split": "dev"})

Saved: M:\Github\TFM_SeismicPhasePicker_ColombianAndes\data\processed\eval_results\records_eqcct_all.pkl  (15566 windows x 2 models)
Saved: M:\Github\TFM_SeismicPhasePicker_ColombianAndes\data\processed\eval_results\records_eqcct_all_dev.pkl  (16414 windows x 2 models)


### 7.1. Clear intermediate chunks

Optional. Clear chunks only after confirming the consolidated records; rebuilding them would require another inference pass.

In [9]:
CLEAR = False

if CLEAR:
    for name in (["test", "dev"] if COLLECT_DEV else ["test"]):
        for label in ("baseline", "finetuned"):
            eu.clear_parts(PARTS / f"{TAG}_{name}" / label)

## 8. Record check

Inspect a saved record and verify the selected split's population counts. This cell does not calculate performance metrics.

In [10]:
r = rec_base[0]
print("keys of one record:", sorted(r.keys()))
print("example:", {k: r[k] for k in ("station", "category", "true")})
print("P peaks (sample, prob):", r["peaks"]["P"][:3])

n_eq    = sum(1 for x in rec_base if x["category"] == "earthquake")
n_noise = sum(1 for x in rec_base if x["category"] == "noise")
n_p     = sum(1 for x in rec_base if x["true"]["P"] is not None)
n_s     = sum(1 for x in rec_base if x["true"]["S"] is not None)
n_synth = sum(1 for x in rec_base if x["category"] == "noise"
              and bool(x.get("trace_synthetic", False)))

print(f"\nwindows: {len(rec_base)}  |  earthquake {n_eq}  noise {n_noise} (synthetic {n_synth})")
print(f"reference arrivals: P in {n_p} windows, S in {n_s}")
print("\nDone. Continue in evaluate_p1_picking.ipynb (no GPU needed).")

keys of one record: ['category', 'distance', 'magnitude', 'peaks', 'snr_p', 'snr_s', 'source_depth_km', 'split', 'station', 'trace_name', 'trace_synthetic', 'true']
example: {'station': 'BRJC', 'category': 'earthquake', 'true': {'P': 3000, 'S': 4264}}
P peaks (sample, prob): [(3004, 0.5221189260482788)]

windows: 15566  |  earthquake 7783  noise 7783 (synthetic 7783)
reference arrivals: P in 7179 windows, S in 5026

Done. Continue in evaluate_p1_picking.ipynb (no GPU needed).
